# 📝 Linguistic Agent - Standalone Training Notebook
This notebook is fully independent and contains all code for automated dataset downloading, Whisper transcription (Phase 1), and BERT classifier training (Phase 2).

In [ ]:
!pip install -q torch torchaudio librosa soundfile scikit-learn matplotlib tqdm kaggle transformers

In [ ]:
import os
import sys
import time
import json
import random
import re
import subprocess
import warnings
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import librosa
from transformers import (
    AutoTokenizer, AutoModel, WhisperProcessor, WhisperForConditionalGeneration, 
    GPT2Tokenizer, GPT2LMHeadModel
)
from sklearn.metrics import roc_curve, auc as sklearn_auc, accuracy_score, f1_score
from scipy.optimize import brentq
from scipy.interpolate import interp1d
from tqdm import tqdm

warnings.filterwarnings("ignore", category=UserWarning)

# ─────────────────────────────────────────────────────────────────────────────
# 1. Configuration & Download
# ─────────────────────────────────────────────────────────────────────────────

DATA_DIR = "data/asvspoof5"
KAGGLE_DATASET = "aniket202411001/asvspoof5-flac"
RESULTS_DIR = Path("results")
CACHE_DIR = Path("cache/transcripts")
DATA_FACTOR = 0.6
BATCH_SIZE = 16
EPOCHS = 10  # Fewer epochs usually needed for fine-tuning BERT heads
LR = 2e-5    # Lower LR for BERT
PATIENCE = 3

def auto_download_dataset(data_dir, dataset_name):
    data_path = Path(data_dir)
    if data_path.exists() and any(data_path.iterdir()):
        print(f"[Dataset] Found at '{data_dir}'.")
        return
    print(f"[Dataset] Downloading from Kaggle into '{data_dir}'...")
    data_path.mkdir(parents=True, exist_ok=True)
    subprocess.run([sys.executable, "-m", "kaggle", "datasets", "download", "-d", dataset_name, "--unzip", "-p", str(data_path)], check=True)

try:
    auto_download_dataset(DATA_DIR, KAGGLE_DATASET)
except Exception as e:
    print(f"Warning: Could not download dataset: {e}")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 2. Audio Loading & Transcription (Phase 1 Setup)
# ─────────────────────────────────────────────────────────────────────────────

def load_audio(path, target_sr=16000, mono=True):
    waveform, sr = librosa.load(str(path), sr=target_sr, mono=mono)
    peak = np.max(np.abs(waveform))
    if peak > 0: waveform = waveform / peak
    return waveform.astype(np.float32), target_sr

def extract_text_features(transcript):
    words = transcript.lower().split() if transcript.strip() else []
    n_words = len(words) + 1e-9
    
    ppl_norm = 0.5 # Placeholder for fast training, could integrate GPT-2 perplexity
    
    rep_rate = 0.0
    if len(words) >= 2:
        bigrams = [f"{words[i]} {words[i+1]}" for i in range(len(words)-1)]
        rep_rate = 1.0 - len(set(bigrams)) / (len(bigrams) + 1e-9)
    
    disfluency_rate = len(re.findall(r'\b(um|uh|er|ah|like|you know)\b', transcript.lower())) / n_words
    ttr = len(set(words)) / n_words
    sent_len_norm = min(n_words / 50.0, 1.0)
    
    return np.array([ppl_norm, rep_rate, disfluency_rate, 1.0-ttr, sent_len_norm], dtype=np.float32)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3. Dataset & Caching Logic
# ─────────────────────────────────────────────────────────────────────────────

class LinguisticDataset(Dataset):
    def __init__(self, root_dir, split="train", data_factor=0.6, tokenizer=None, max_length=512):
        self.root = Path(root_dir)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.items = []
        
        split_dir = self.root / split
        if not split_dir.exists():
            for c in ["flac_T", "flac_D", "flac_E_eval"]:
                if (self.root / c).exists() and split in c.lower():
                    split_dir = self.root / c; break
        
        bona_dir = split_dir / "bonafide"
        spoof_dir = split_dir / "spoof"
        
        bona_files = list(bona_dir.glob("**/*.flac")) + list(bona_dir.glob("**/*.wav"))
        spoof_files = list(spoof_dir.glob("**/*.flac")) + list(spoof_dir.glob("**/*.wav"))
        
        if len(bona_files) > 0:
            k_bona = max(1, int(len(bona_files) * data_factor))
            bona_files = random.sample(bona_files, min(len(bona_files), k_bona))
        else:
            bona_files = []
            
        if len(spoof_files) > 0:
            k_spoof = max(1, int(len(spoof_files) * data_factor))
            spoof_files = random.sample(spoof_files, min(len(spoof_files), k_spoof))
        else:
            spoof_files = []
        
        for f in bona_files: self.items.append((f, 0))
        for f in spoof_files: self.items.append((f, 1))
        
        print(f"Loaded {len(self.items)} audio files for {split}")

    def __len__(self): return len(self.items)
    
    def __getitem__(self, idx):
        path, label = self.items[idx]
        
        # Check for cached transcript
        cache_file = CACHE_DIR / f"{path.stem}.txt"
        if cache_file.exists():
            with open(cache_file, "r", encoding="utf-8") as f:
                transcript = f.read().strip()
        else:
            transcript = ""  # Will be handled in Phase 1 if empty, but for robust training we assume cached
            
        hand_feats = extract_text_features(transcript)
        
        if self.tokenizer:
            enc = self.tokenizer(transcript, truncation=True, max_length=self.max_length, padding="max_length", return_tensors="pt")
            input_ids = enc["input_ids"].squeeze(0)
            attention_mask = enc["attention_mask"].squeeze(0)
        else:
            input_ids = torch.empty(0)
            attention_mask = torch.empty(0)
            
        return input_ids, attention_mask, torch.from_numpy(hand_feats), torch.tensor(label, dtype=torch.float32)

def build_transcript_cache(dataset_splits):
    """Phase 1: Transcribe all audio using Whisper and cache it to disk to save massive time during epochs."""
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    processor = WhisperProcessor.from_pretrained("openai/whisper-small")
    model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to(device).eval()
    
    for split_name, ds in dataset_splits.items():
        print(f"\nBuilding cache for {split_name} ({len(ds)} files)...")
        for path, _ in tqdm(ds.items, desc=f"Transcribing {split_name}"):
            cache_file = CACHE_DIR / f"{path.stem}.txt"
            if cache_file.exists():
                continue
            try:
                wav, sr = load_audio(path)
                inputs = processor(wav, sampling_rate=16000, return_tensors="pt").input_features.to(device)
                with torch.no_grad():
                    predicted_ids = model.generate(inputs)
                transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
                with open(cache_file, "w", encoding="utf-8") as f:
                    f.write(transcription)
            except Exception as e:
                with open(cache_file, "w", encoding="utf-8") as f:
                    f.write("")  # Write empty on failure to prevent infinite retries
    
    print("\nTranscription Phase 1 Complete. Moving to BERT Training.")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4. Model Architecture (BERT + Hand-crafted Features)
# ─────────────────────────────────────────────────────────────────────────────

class LinguisticClassifier(nn.Module):
    def __init__(self, bert_name="bert-base-uncased", dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(bert_name)
        
        # Freeze base BERT layers to speed up training if desired, but we'll leave it unfrozen for fine-tuning
        
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.bert.config.hidden_size + 5, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )
    def forward(self, input_ids, attention_mask, hand_features):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :] # Use CLS token
        x = torch.cat([cls, hand_features], dim=-1)
        return torch.sigmoid(self.head(x).squeeze(-1))


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 5. Evaluation & Utilities
# ─────────────────────────────────────────────────────────────────────────────

def compute_eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores, pos_label=1)
    fnr = 1 - tpr
    try:
        return brentq(lambda x: interp1d(fpr, fnr - fpr)(x), 0, 1)
    except:
        return float(np.mean(np.abs(fnr - fpr)))

def save_results(dir_path, eer, auc, acc, f1):
    dir_path = Path(dir_path)
    dir_path.mkdir(parents=True, exist_ok=True)
    with open(dir_path / "results.txt", "w") as f:
        f.write(f"EER: {eer*100:.4f}%\nAUC: {auc:.4f}\nAccuracy: {acc*100:.4f}%\nF1: {f1:.4f}\n")
    with open(dir_path / "results.json", "w") as f:
        json.dump({"eer": eer, "auc": auc, "accuracy": acc, "f1": f1}, f, indent=4)
    print(f"Results saved to {dir_path}")

class Timer:
    def __init__(self, epochs, steps):
        self.epochs, self.steps, self.start = epochs, steps, time.time()
    def step(self, e, s, loss):
        elapsed = time.time() - self.start
        total_steps = self.epochs * self.steps
        current_step = e * self.steps + s
        eta = elapsed / max(1, current_step + 1) * (total_steps - current_step - 1)
        return f"\rEpoch {e+1}/{self.epochs} | Step {s+1}/{self.steps} | Loss: {loss:.4f} | ETA: {int(eta//60)}m {int(eta%60)}s   "


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6. Main Training Loop
# ─────────────────────────────────────────────────────────────────────────────

def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    print("Scanning datasets...")
    # We initialize datasets without tokenizer first to build the cache
    train_ds_raw = LinguisticDataset(DATA_DIR, "train", data_factor=DATA_FACTOR)
    val_ds_raw = LinguisticDataset(DATA_DIR, "val", data_factor=DATA_FACTOR)
    test_ds_raw = LinguisticDataset(DATA_DIR, "test", data_factor=DATA_FACTOR)
    
    if len(train_ds_raw) == 0: 
        print("No training data found. Please check DATA_DIR.")
        return

    # PHASE 1: Build Transcript Cache
    build_transcript_cache({"train": train_ds_raw, "val": val_ds_raw, "test": test_ds_raw})
    
    # PHASE 2: BERT Training
    print("\nInitializing BERT tokenizer and model...")
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    
    train_ds = LinguisticDataset(DATA_DIR, "train", data_factor=DATA_FACTOR, tokenizer=tokenizer)
    val_ds = LinguisticDataset(DATA_DIR, "val", data_factor=DATA_FACTOR, tokenizer=tokenizer)
    test_ds = LinguisticDataset(DATA_DIR, "test", data_factor=DATA_FACTOR, tokenizer=tokenizer)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

    model = LinguisticClassifier("bert-base-uncased").to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    criterion = nn.BCELoss()
    
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    best_eer = float('inf')
    patience_ctr = 0
    
    timer = Timer(EPOCHS, len(train_loader))
    
    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0.0
        for i, (input_ids, attention_mask, hand_feats, labels) in enumerate(train_loader):
            input_ids, attention_mask, hand_feats, labels = input_ids.to(device), attention_mask.to(device), hand_feats.to(device), labels.to(device)
            
            optimizer.zero_grad()
            preds = model(input_ids, attention_mask, hand_feats)
            loss = criterion(preds, labels)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            print(timer.step(epoch, i, loss.item()), end="")
            
        print()
        
        # Validation
        model.eval()
        val_labels, val_scores = [], []
        with torch.no_grad():
            for input_ids, attention_mask, hand_feats, labels in val_loader:
                input_ids, attention_mask, hand_feats = input_ids.to(device), attention_mask.to(device), hand_feats.to(device)
                preds = model(input_ids, attention_mask, hand_feats)
                val_scores.extend(preds.cpu().numpy())
                val_labels.extend(labels.numpy())
                
        val_labels, val_scores = np.array(val_labels), np.array(val_scores)
        eer = compute_eer(val_labels, val_scores)
        print(f"Epoch {epoch+1} Validation EER: {eer*100:.2f}%")
        
        if eer < best_eer:
            best_eer = eer
            patience_ctr = 0
            torch.save({"model_state_dict": model.state_dict()}, RESULTS_DIR / "best.pt")
            print(" -> Saved new best model!")
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print("Early stopping triggered.")
                break

    # Final Evaluation
    print("\nRunning final evaluation on Test Set...")
    model.load_state_dict(torch.load(RESULTS_DIR / "best.pt")["model_state_dict"])
    model.eval()
    test_labels, test_scores = [], []
    with torch.no_grad():
        for input_ids, attention_mask, hand_feats, labels in test_loader:
            input_ids, attention_mask, hand_feats = input_ids.to(device), attention_mask.to(device), hand_feats.to(device)
            preds = model(input_ids, attention_mask, hand_feats)
            test_scores.extend(preds.cpu().numpy())
            test_labels.extend(labels.numpy())
            
    test_labels, test_scores = np.array(test_labels), np.array(test_scores)
    test_preds = (test_scores >= 0.5).astype(int)
    
    eer = compute_eer(test_labels, test_scores)
    auc = sklearn_auc(*roc_curve(test_labels, test_scores, pos_label=1)[:2])
    acc = accuracy_score(test_labels, test_preds)
    f1 = f1_score(test_labels, test_preds)
    
    save_results(RESULTS_DIR, eer, auc, acc, f1)

if __name__ == "__main__":
    train()
